# Preprocessing Technique: Feature Engineering — Dimension Reduction (PCA)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# This notebook is 6th and LAST in the run order - it loads Member 4's scaled output.
# By this point the data is already fully numeric and scaled, thanks to every
# teammate's step before this one.
df = pd.read_csv("output_after_scaling.csv")
print("Raw shape:", df.shape)

Raw shape: (100000, 30)


### Step 1: Collect the data / build a numeric feature set


In [2]:
# Use every remaining feature column except the target - by this point ALL of them
# are already numeric and scaled, thanks to the previous 5 teammates' steps, so no
# column needs to be hand-picked the way the standalone version of this notebook did.
target = df["satisfaction_level"]
features_for_pca = df.drop(columns=["satisfaction_level"])
features_for_pca.head()

,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
0,0.775324,0.590069,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.427281
1,-0.226447,-1.059138,1,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,1.482789
2,0.357919,1.189361,2,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.577940
3,-0.309928,-1.086947,1,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.379714
4,1.359690,-0.530760,2,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,1.519129


### Step 2: Standardize features (mean=0, std=1)

In [3]:
# ticket_price/spend_amount were already scaled by Member 4; the one-hot and encoded
# columns are already numeric. PCA still needs everything on a comparable scale, so we
# standardize the full feature set here (re-scaling already-scaled columns is harmless).
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(features_for_pca),
    columns=features_for_pca.columns
)
print("Standardized data (mean ~0, std ~1):")
X_scaled.describe().round(2)

Standardized data (mean ~0, std ~1):


,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
count,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,...,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00,100000.00
mean,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.31,-1.42,-2.73,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-1.73
25%,-0.73,-0.88,-0.84,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-0.74
50%,-0.14,-0.16,-0.84,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,-0.11
75%,0.61,0.81,1.05,-0.11,-0.12,-0.05,-0.08,-0.23,-0.38,-0.18,...,-0.27,-0.08,-0.08,-0.13,-0.05,-0.09,-0.05,-0.14,-0.13,0.58
max,2.61,3.34,1.05,9.14,8.39,21.59,11.86,4.44,2.63,5.70,...,3.66,11.85,11.83,7.79,20.74,11.65,21.15,6.92,7.85,2.65


### Step 4 & 5: Find eigenvectors (new directions) and rank them by eigenvalues

In [4]:
pca_full = PCA()
pca_full.fit(X_scaled)

# Eigenvalues: how much variance each component captures (sklearn stores these directly)
eigenvalues = pca_full.explained_variance_
explained_var_pct = pca_full.explained_variance_ratio_ * 100
cumulative_pct = np.cumsum(explained_var_pct)

eigen_table = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(eigenvalues))],
    "eigenvalue": eigenvalues.round(3),
    "explained_variance_%": explained_var_pct.round(1),
    "cumulative_%": cumulative_pct.round(1)
})
eigen_table

,component,eigenvalue,explained_variance_%,cumulative_%
0,PC1,2.052,7.1,7.1
1,PC2,1.455,5.0,12.1
2,PC3,1.359,4.7,16.8
3,PC4,1.116,3.8,20.6
4,PC5,1.102,3.8,24.4
5,PC6,1.066,3.7,28.1
6,PC7,1.051,3.6,31.7
7,PC8,1.041,3.6,35.3
8,PC9,1.031,3.6,38.9
9,PC10,1.026,3.5,42.4


In [5]:
# Eigenvectors: the loadings - each component's weights on the original features
eigenvectors = pd.DataFrame(
    pca_full.components_,
    columns=features_for_pca.columns,
    index=[f"PC{i+1}" for i in range(len(eigenvalues))]
)
eigenvectors.round(2)

,ticket_price,spend_amount,attraction_level_encoded,attraction_category_Botanical Garden,attraction_category_Cheng Shi Gong Yuan,attraction_category_Food Street,attraction_category_Gong Ye Lv You,attraction_category_Historical Architecture,attraction_category_Historical Culture,attraction_category_Historical Site,...,attraction_category_Theme Park,attraction_category_Urban Landmark,attraction_category_Urban Landscape,attraction_category_Wen Bo Yuan Guan,attraction_category_Wen Hua Lv You,attraction_category_Wen Hua Yi Shu,attraction_category_Zhu Ti Zhan Lan,attraction_category_Zi Ran Qi Guan,attraction_category_Zoo,province_target_enc
PC1,0.60,0.38,0.42,-0.08,-0.14,-0.10,-0.04,-0.06,-0.13,-0.01,...,0.34,0.08,-0.09,-0.17,0.01,-0.12,0.03,-0.01,-0.04,0.23
PC2,-0.18,-0.18,0.24,-0.11,-0.03,-0.08,-0.06,0.02,-0.09,-0.12,...,-0.26,-0.19,-0.01,-0.05,0.03,-0.12,-0.17,-0.11,-0.23,0.50
PC3,0.11,0.10,-0.28,0.02,-0.04,0.08,0.02,-0.18,-0.35,-0.05,...,-0.04,0.08,-0.05,-0.01,-0.03,0.04,0.10,-0.03,0.10,-0.24
PC4,0.02,0.01,0.10,-0.07,-0.14,-0.13,-0.11,-0.29,0.79,-0.04,...,-0.13,0.03,-0.06,-0.19,-0.11,-0.05,0.02,-0.04,-0.03,-0.09
PC5,-0.01,0.08,-0.25,-0.04,0.06,0.24,0.18,0.17,0.23,-0.20,...,0.46,-0.10,-0.06,0.18,0.27,-0.07,-0.07,-0.13,-0.10,0.23
PC6,0.02,-0.00,-0.06,0.01,-0.00,0.09,0.05,-0.16,0.08,-0.07,...,-0.18,0.03,-0.03,0.04,0.05,0.01,0.04,-0.04,0.04,0.01
PC7,0.02,-0.02,-0.00,-0.02,-0.05,-0.01,0.00,0.73,0.02,-0.02,...,-0.18,0.02,-0.03,-0.04,0.03,-0.04,0.00,-0.01,-0.02,0.01
PC8,-0.01,0.00,-0.07,0.02,0.15,0.19,0.14,-0.43,-0.01,-0.27,...,0.06,-0.11,0.02,0.28,0.17,0.01,-0.06,-0.09,-0.06,0.05
PC9,-0.02,-0.00,0.02,0.00,0.15,-0.04,-0.03,-0.16,-0.04,0.79,...,0.16,-0.17,0.10,0.13,-0.01,0.02,-0.12,-0.00,-0.18,0.01
PC10,0.03,0.01,-0.06,-0.11,-0.22,0.20,0.23,-0.02,0.04,0.42,...,-0.32,0.18,-0.23,-0.02,0.30,-0.20,0.08,-0.30,0.01,0.05


In [6]:
for pc in eigenvectors.index[:2]:
    top_features = eigenvectors.loc[pc].abs().sort_values(ascending=False)
    print(f"{pc} — features ranked by loading magnitude:")
    print(top_features.round(2))
    print()

PC1 — features ranked by loading magnitude:
ticket_price                                   0.60
attraction_level_encoded                       0.42
spend_amount                                   0.38
attraction_category_Theme Park                 0.34
province_target_enc                            0.23
attraction_category_Wen Bo Yuan Guan           0.17
attraction_category_Cheng Shi Gong Yuan        0.14
attraction_category_Historical Culture         0.13
attraction_category_Wen Hua Yi Shu             0.12
attraction_category_Food Street                0.10
attraction_category_Urban Landscape            0.09
attraction_category_Natural Culture            0.09
attraction_category_Revolutionary Site         0.09
attraction_category_Urban Landmark             0.08
attraction_category_Botanical Garden           0.08
attraction_category_Religious Culture          0.07
attraction_category_Sports & Leisure           0.06
attraction_category_Min Su Wen Hua             0.06
attraction_category_

### Step 6: Keep the top components (reduce dimensions)

In [7]:
n_components_90 = np.argmax(cumulative_pct >= 90) + 1
print(f"Components needed to retain 90% of variance: {n_components_90} (out of {X_scaled.shape[1]} original features)")

pca_final = PCA(n_components=n_components_90)
X_reduced = pca_final.fit_transform(X_scaled)
reduced_df = pd.DataFrame(X_reduced, columns=[f"PC{i+1}" for i in range(n_components_90)])
print("Reduced dataset shape:", reduced_df.shape)
reduced_df.head()

Components needed to retain 90% of variance: 24 (out of 29 original features)
Reduced dataset shape: (100000, 24)


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC15,PC16,PC17,PC18,PC19,PC20,PC21,PC22,PC23,PC24
0,0.421084,-1.309662,-0.354689,-0.159513,-1.963282,-1.776185,1.216489,3.100193,-0.419602,0.643121,...,0.188359,-0.260233,-0.239338,-0.136154,-0.080446,-0.202149,-0.041775,0.080752,-0.118303,-0.036255
1,-1.332975,-0.409108,-1.088941,2.226674,0.808738,0.269927,0.069983,0.011668,-0.138417,0.172170,...,-0.014660,-0.031940,-0.047592,-0.036120,-0.013725,-0.048147,0.005879,0.010551,-0.015387,-0.005119
2,1.422851,0.532844,-0.630894,-0.054753,0.143908,-0.049085,-0.006435,0.013128,0.026920,0.027160,...,0.005942,0.014621,0.005518,0.019897,-0.003082,0.026151,-0.020085,-0.016595,0.011112,0.006278
3,-1.364300,-0.925003,0.340326,-0.006347,-0.258674,0.016008,0.002526,-0.034372,-0.018134,-0.044673,...,-0.008896,-0.003458,0.008026,-0.012596,0.006764,-0.012811,0.021307,0.013497,-0.005577,-0.003147
4,1.511910,0.353873,-1.623823,-0.982954,-1.096606,3.035668,-0.192669,0.119947,-0.247967,-0.045276,...,0.058803,-0.155742,-0.138016,-0.099486,-0.007574,-0.137054,-0.013810,0.035066,-0.084232,-0.024461


## Final Output — the Fully Processed Dataset


In [8]:
final_df = pd.DataFrame(X_reduced, columns=[f"PC{i+1}" for i in range(n_components_90)])
final_df["satisfaction_level"] = target.values

final_df.to_csv("final_processed_dataset.csv", index=False)
print("Saved final_processed_dataset.csv - shape:", final_df.shape)
print("This is the finished output of the whole team's chained pipeline.")

Saved final_processed_dataset.csv - shape: (100000, 25)
This is the finished output of the whole team's chained pipeline.
